# Dishify Scoring and Ranking Demo

This notebook shows a simple, deterministic scoring pipeline for Dishify.
It applies hard filters first, then computes a weighted score for each recipe.
LLMs are only used later for explanations or optional re-ranking of top results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_colwidth", 120)

## 1) Create toy recipe data

We use a small in-memory dataset with fields that mirror Dishify's pipeline inputs.

In [ ]:
recipes = [
    {
        "title": "Chicken Rice Bowl",
        "ingredients": ["chicken", "rice", "soy sauce", "garlic", "ginger"],
        "cooking_time_minutes": 25,
        "cuisine": "asian",
        "tags": ["high_protein", "quick"],
        "qdrant_similarity_score": 0.82,
        "quality_score": 0.74,
    },
    {
        "title": "Tomato Pasta",
        "ingredients": ["tomato", "pasta", "garlic", "olive oil", "basil"],
        "cooking_time_minutes": 20,
        "cuisine": "italian",
        "tags": ["vegetarian", "quick"],
        "qdrant_similarity_score": 0.78,
        "quality_score": 0.71,
    },
    {
        "title": "Veggie Stir Fry",
        "ingredients": ["broccoli", "carrot", "zucchini", "soy sauce", "garlic"],
        "cooking_time_minutes": 18,
        "cuisine": "asian",
        "tags": ["vegetarian", "quick"],
        "qdrant_similarity_score": 0.62,
        "quality_score": 0.68,
    },
    {
        "title": "Spicy Chicken Tacos",
        "ingredients": ["chicken", "tortilla", "tomato", "onion", "chili"],
        "cooking_time_minutes": 30,
        "cuisine": "mexican",
        "tags": ["spicy", "high_protein"],
        "qdrant_similarity_score": 0.84,
        "quality_score": 0.76,
    },
    {
        "title": "Creamy Mushroom Pasta",
        "ingredients": ["mushroom", "pasta", "cream", "garlic"],
        "cooking_time_minutes": 35,
        "cuisine": "italian",
        "tags": ["vegetarian"],
        "qdrant_similarity_score": 0.75,
        "quality_score": 0.69,
    },
    {
        "title": "Egg Fried Rice",
        "ingredients": ["egg", "rice", "soy sauce", "onion", "peas"],
        "cooking_time_minutes": 15,
        "cuisine": "asian",
        "tags": ["quick"],
        "qdrant_similarity_score": 0.70,
        "quality_score": 0.66,
    },
    {
        "title": "Lentil Curry",
        "ingredients": ["lentil", "tomato", "onion", "curry powder", "garlic"],
        "cooking_time_minutes": 40,
        "cuisine": "indian",
        "tags": ["vegetarian", "spicy", "high_protein"],
        "qdrant_similarity_score": 0.60,
        "quality_score": 0.72,
    },
    {
        "title": "Salmon Salad",
        "ingredients": ["salmon", "lettuce", "tomato", "cucumber", "olive oil"],
        "cooking_time_minutes": 20,
        "cuisine": "mediterranean",
        "tags": ["high_protein", "quick"],
        "qdrant_similarity_score": 0.58,
        "quality_score": 0.70,
    },
    {
        "title": "Tomato Zucchini Skillet",
        "ingredients": ["tomato", "zucchini", "onion", "olive oil"],
        "cooking_time_minutes": 22,
        "cuisine": "mediterranean",
        "tags": ["vegetarian", "quick"],
        "qdrant_similarity_score": 0.55,
        "quality_score": 0.65,
    },
 ]

recipes_df = pd.DataFrame(recipes)
recipes_df

## 2) Define user input

These inputs simulate what the user provides or selects in the app.

In [ ]:
user_available_ingredients = ["chicken", "rice", "tomato", "onion"]
user_excluded_ingredients = ["mushroom"]
preferred_max_time = 30
user_preference_tags = ["spicy", "high_protein"]
preferred_cuisines = ["asian", "mexican"]

user_available_set = {item.lower() for item in user_available_ingredients}
user_excluded_set = {item.lower() for item in user_excluded_ingredients}
user_preference_set = {item.lower() for item in user_preference_tags}
preferred_cuisine_set = {item.lower() for item in preferred_cuisines}

user_available_ingredients, user_excluded_ingredients, preferred_max_time, user_preference_tags, preferred_cuisines

## 3) Hard filtering

Hard filters remove recipes that violate strict constraints (like excluded ingredients).

In [ ]:
def hard_filter(df: pd.DataFrame, excluded_set: set[str]) -> pd.DataFrame:
    def _passes(recipe_ingredients: list[str]) -> bool:
        recipe_set = {item.lower() for item in recipe_ingredients}
        return recipe_set.isdisjoint(excluded_set)
    return df[df["ingredients"].apply(_passes)].reset_index(drop=True)

print("Before filtering:")
print(recipes_df[["title", "ingredients"]])

filtered_df = hard_filter(recipes_df, user_excluded_set)

print("\nAfter filtering:")
print(filtered_df[["title", "ingredients"]])

## 4) Ingredient scoring

We score ingredient fit using both user coverage and recipe coverage.

In [ ]:
def ingredient_fit(recipe_ingredients: list[str], user_ingredients: set[str]) -> dict:
    recipe_set = {item.lower() for item in recipe_ingredients}
    matched = sorted(recipe_set.intersection(user_ingredients))
    missing = sorted(user_ingredients.difference(recipe_set))

    available_coverage = len(matched) / max(len(user_ingredients), 1)
    recipe_coverage = len(matched) / max(len(recipe_set), 1)
    score = 0.70 * available_coverage + 0.30 * recipe_coverage
    return {
        "matched_ingredients": matched,
        "missing_ingredients": missing,
        "ingredient_fit": score,
    }

ingredient_scores = filtered_df["ingredients"].apply(
    lambda items: ingredient_fit(items, user_available_set)
)
ingredient_scores_df = pd.DataFrame(list(ingredient_scores))

scored_df = pd.concat([filtered_df.reset_index(drop=True), ingredient_scores_df], axis=1)
scored_df[["title", "matched_ingredients", "missing_ingredients", "ingredient_fit"]]

## 5) Semantic similarity

We reuse the stored Qdrant similarity score and clamp it to 0–1.

In [ ]:
def clamp01(value: float) -> float:
    return float(np.clip(value, 0.0, 1.0))

scored_df["semantic_similarity"] = scored_df["qdrant_similarity_score"].apply(clamp01)
scored_df[["title", "semantic_similarity"]]

## 6) Preference fit

Preferences match on recipe tags or cuisine. If no preferences are provided, we return 0.5.

In [ ]:
def preference_fit(tags: list[str], cuisine: str) -> float:
    preferences = user_preference_set.union(preferred_cuisine_set)
    if not preferences:
        return 0.5
    recipe_tags = {item.lower() for item in tags}
    recipe_cuisine = cuisine.lower()
    matches = len(preferences.intersection(recipe_tags.union({recipe_cuisine})))
    return matches / len(preferences)

scored_df["preference_fit"] = scored_df.apply(
    lambda row: preference_fit(row["tags"], row["cuisine"]), axis=1
)
scored_df[["title", "preference_fit"]]

## 7) Time fit

Shorter recipes score higher relative to the preferred maximum time.

In [ ]:
def time_fit(recipe_time: int | float | None, preferred_time: int | float | None) -> float:
    if preferred_time is None:
        return 0.5
    if recipe_time is None:
        return 0.5
    if recipe_time <= preferred_time:
        return 1.0
    return clamp01(preferred_time / recipe_time)

scored_df["time_fit"] = scored_df["cooking_time_minutes"].apply(
    lambda value: time_fit(value, preferred_max_time)
)
scored_df[["title", "time_fit"]]

## 8) Quality fit

Quality can represent ratings, reviews, or source reliability. We compute it separately.

In [ ]:
scored_df["quality_fit"] = scored_df["quality_score"].apply(clamp01)
scored_df[["title", "quality_fit"]]

## 9) Final scoring

We combine the main signals into a final score. In this toy example, quality is computed but not weighted.

In [ ]:
scored_df["final_score"] = (
    0.45 * scored_df["ingredient_fit"]
    + 0.30 * scored_df["semantic_similarity"]
    + 0.15 * scored_df["preference_fit"]
    + 0.10 * scored_df["time_fit"]
)

result_cols = [
    "title",
    "matched_ingredients",
    "missing_ingredients",
    "ingredient_fit",
    "semantic_similarity",
    "preference_fit",
    "time_fit",
    "quality_fit",
    "final_score",
]

ranked_df = scored_df[result_cols].sort_values(
    by="final_score", ascending=False
).reset_index(drop=True)

ranked_df

## 10) Explanation output

Rule-based text explains why the top results rank well.

In [ ]:
def explain_row(row: pd.Series) -> str:
    matched = row["matched_ingredients"]
    match_count = len(matched)
    reasons = [f"uses {match_count} of your available ingredients"]
    if row["semantic_similarity"] >= 0.7:
        reasons.append("has high semantic similarity")
    if row["preference_fit"] > 0:
        reasons.append("matches your preferences or cuisine")
    if row["time_fit"] >= 1.0:
        reasons.append(f"can be cooked within {preferred_max_time} minutes")
    return "This recipe ranks highly because it " + ", ".join(reasons) + "."

top_3 = ranked_df.head(3).copy()
top_3["explanation"] = top_3.apply(explain_row, axis=1)
top_3[["title", "final_score", "explanation"]]

## 11) Visualization

A simple bar chart of final scores.

In [ ]:
plt.figure(figsize=(10, 4))
plt.bar(ranked_df["title"], ranked_df["final_score"], color="#4C78A8")
plt.title("Dishify Final Scores")
plt.ylabel("final_score")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [1]:
def ingredient_fit_score(user_ingredients: list[str], recipe_ingredients: list[str]) -> dict:
    user_set = {item.lower() for item in user_ingredients}
    recipe_set = {item.lower() for item in recipe_ingredients}
    matched = sorted(user_set.intersection(recipe_set))
    available_coverage = len(matched) / max(len(user_set), 1)
    recipe_coverage = len(matched) / max(len(recipe_set), 1)
    score = 0.70 * available_coverage + 0.30 * recipe_coverage
    return {
        "matched": matched,
        "available_coverage": available_coverage,
        "recipe_coverage": recipe_coverage,
        "ingredient_fit": score,
    }

a = ["chicken", "rice", "tomato", "onion"]
b = ["chicken", "rice", "garlic", "soy sauce"]
c = ["tomato", "basil", "olive oil"]

pairs = [("A vs B", a, b), ("A vs C", a, c), ("B vs C", b, c)]

for label, left, right in pairs:
    result = ingredient_fit_score(left, right)
    print(label)
    print(f"  matched: {result['matched']}")
    print(f"  available_coverage: {result['available_coverage']:.3f}")
    print(f"  recipe_coverage: {result['recipe_coverage']:.3f}")
    print(f"  ingredient_fit: {result['ingredient_fit']:.3f}\n")

A vs B
  matched: ['chicken', 'rice']
  available_coverage: 0.500
  recipe_coverage: 0.500
  ingredient_fit: 0.500

A vs C
  matched: ['tomato']
  available_coverage: 0.250
  recipe_coverage: 0.333
  ingredient_fit: 0.275

B vs C
  matched: []
  available_coverage: 0.000
  recipe_coverage: 0.000
  ingredient_fit: 0.000



## 12) Conclusion

- Hard filters enforce strict constraints like excluded ingredients.
- Weighted scoring captures soft preferences and ranking signals.
- Qdrant provides semantic similarity, while preference and time scores are deterministic.
- An LLM can later explain top results, but should not replace the ranking logic.